<a href="https://colab.research.google.com/github/Swaraj-sj2000/Machine-leaning-projects/blob/main/rev2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [110]:
import tensorflow as tf
print("TensorFlow:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))


TensorFlow: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [111]:
import json
kaggle_dict={'username':'sjrohan',
             'key':'KGAT_931103ac2d60512613022382987768cb'}
with open('kaggle.json','w') as f:
  json.dump(kaggle_dict,f)

In [112]:
!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json


In [113]:
!pip install -q kaggle

In [114]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

movies_df= kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "tmdb/tmdb-movie-metadata",
    "tmdb_5000_movies.csv"
)



/tmp/ipykernel_7227/3358068137.py:4: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  movies_df= kagglehub.load_dataset(


Using Colab cache for faster access to the 'tmdb-movie-metadata' dataset.


In [115]:
credits_df= kagglehub.load_dataset(
    KaggleDatasetAdapter.PANDAS,
    "tmdb/tmdb-movie-metadata",
    "tmdb_5000_credits.csv"
)



/tmp/ipykernel_7227/3042225847.py:1: DeprecationWarning: Use dataset_load() instead of load_dataset(). load_dataset() will be removed in a future version.
  credits_df= kagglehub.load_dataset(


Using Colab cache for faster access to the 'tmdb-movie-metadata' dataset.


In [143]:
import numpy as np
import pandas as pd
movies=movies_df.copy()
credits=credits_df.copy()

In [117]:
print(movies.shape)
print(credits.shape)

(4803, 20)
(4803, 4)


In [145]:
movies=movies.merge(credits,on='title')

In [119]:
print(movies.shape)

(4809, 23)


In [120]:
movies.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   budget                4809 non-null   int64  
 1   genres                4809 non-null   object 
 2   homepage              1713 non-null   object 
 3   id                    4809 non-null   int64  
 4   keywords              4809 non-null   object 
 5   original_language     4809 non-null   object 
 6   original_title        4809 non-null   object 
 7   overview              4806 non-null   object 
 8   popularity            4809 non-null   float64
 9   production_companies  4809 non-null   object 
 10  production_countries  4809 non-null   object 
 11  release_date          4808 non-null   object 
 12  revenue               4809 non-null   int64  
 13  runtime               4807 non-null   float64
 14  spoken_languages      4809 non-null   object 
 15  status               

In [146]:
movies.drop(columns=['budget','homepage','production_countries','original_title','popularity','production_companies','production_countries','runtime','spoken_languages','status','tagline','movie_id'],inplace=True)

In [122]:
print(movies.shape)
print(movies.info())

(4809, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4809 entries, 0 to 4808
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   genres             4809 non-null   object 
 1   id                 4809 non-null   int64  
 2   keywords           4809 non-null   object 
 3   original_language  4809 non-null   object 
 4   overview           4806 non-null   object 
 5   release_date       4808 non-null   object 
 6   revenue            4809 non-null   int64  
 7   title              4809 non-null   object 
 8   vote_average       4809 non-null   float64
 9   vote_count         4809 non-null   int64  
 10  cast               4809 non-null   object 
 11  crew               4809 non-null   object 
dtypes: float64(1), int64(3), object(8)
memory usage: 451.0+ KB
None


In [123]:
movies.isnull().sum()

,0
genres,0
id,0
keywords,0
original_language,0
overview,3
release_date,1
revenue,0
title,0
vote_average,0
vote_count,0


In [147]:
movies.dropna(subset=['overview','release_date'],inplace=True)

In [125]:
movies.duplicated().sum()

np.int64(0)

In [126]:
print(movies.iloc[0]['genres'])

[{"id": 28, "name": "Action"}, {"id": 12, "name": "Adventure"}, {"id": 14, "name": "Fantasy"}, {"id": 878, "name": "Science Fiction"}]


In [148]:
import ast
cols = ['genres','keywords','cast','crew']
for col in cols:
  movies[col] = movies[col].apply(ast.literal_eval)

In [149]:
movies.head(2)

,genres,id,keywords,original_language,overview,release_date,revenue,title,vote_average,vote_count,cast,crew
0,"[{'id': 28, 'name': 'Action'}, {'id': 12, 'nam...",19995,"[{'id': 1463, 'name': 'culture clash'}, {'id':...",en,"In the 22nd century, a paraplegic Marine is di...",2009-12-10,2787965087,Avatar,7.2,11800,"[{'cast_id': 242, 'character': 'Jake Sully', '...","[{'credit_id': '52fe48009251416c750aca23', 'de..."
1,"[{'id': 12, 'name': 'Adventure'}, {'id': 14, '...",285,"[{'id': 270, 'name': 'ocean'}, {'id': 726, 'na...",en,"Captain Barbossa, long believed to be dead, ha...",2007-05-19,961000000,Pirates of the Caribbean: At World's End,6.9,4500,"[{'cast_id': 4, 'character': 'Captain Jack Spa...","[{'credit_id': '52fe4232c3a36847f800b579', 'de..."


In [150]:
movies['genres']=movies['genres'].apply(lambda x:[item['name'].replace(' ','_')  for item in x] if isinstance(x,list) else [])
movies['keywords']=movies['keywords'].apply(lambda x:[item['name'].replace(' ','_') for item in x] if isinstance(x,list) else [])
movies['cast']=movies['cast'].apply(lambda x:[item['name'].replace(' ','_') for item in x] if isinstance(x,list) else [][:3])



In [151]:
movies.head()

,genres,id,keywords,original_language,overview,release_date,revenue,title,vote_average,vote_count,cast,crew
0,"[Action, Adventure, Fantasy, Science_Fiction]",19995,"[culture_clash, future, space_war, space_colon...",en,"In the 22nd century, a paraplegic Marine is di...",2009-12-10,2787965087,Avatar,7.2,11800,"[Sam_Worthington, Zoe_Saldana, Sigourney_Weave...","[{'credit_id': '52fe48009251416c750aca23', 'de..."
1,"[Adventure, Fantasy, Action]",285,"[ocean, drug_abuse, exotic_island, east_india_...",en,"Captain Barbossa, long believed to be dead, ha...",2007-05-19,961000000,Pirates of the Caribbean: At World's End,6.9,4500,"[Johnny_Depp, Orlando_Bloom, Keira_Knightley, ...","[{'credit_id': '52fe4232c3a36847f800b579', 'de..."
2,"[Action, Adventure, Crime]",206647,"[spy, based_on_novel, secret_agent, sequel, mi...",en,A cryptic message from Bond’s past sends him o...,2015-10-26,880674609,Spectre,6.3,4466,"[Daniel_Craig, Christoph_Waltz, Léa_Seydoux, R...","[{'credit_id': '54805967c3a36829b5002c41', 'de..."
3,"[Action, Crime, Drama, Thriller]",49026,"[dc_comics, crime_fighter, terrorist, secret_i...",en,Following the death of District Attorney Harve...,2012-07-16,1084939099,The Dark Knight Rises,7.6,9106,"[Christian_Bale, Michael_Caine, Gary_Oldman, A...","[{'credit_id': '52fe4781c3a36847f81398c3', 'de..."
4,"[Action, Adventure, Science_Fiction]",49529,"[based_on_novel, mars, medallion, space_travel...",en,"John Carter is a war-weary, former military ca...",2012-03-07,284139100,John Carter,6.1,2124,"[Taylor_Kitsch, Lynn_Collins, Samantha_Morton,...","[{'credit_id': '52fe479ac3a36847f813eaa3', 'de..."


In [152]:
print(movies['cast'].head())

0    [Sam_Worthington, Zoe_Saldana, Sigourney_Weave...
1    [Johnny_Depp, Orlando_Bloom, Keira_Knightley, ...
2    [Daniel_Craig, Christoph_Waltz, Léa_Seydoux, R...
3    [Christian_Bale, Michael_Caine, Gary_Oldman, A...
4    [Taylor_Kitsch, Lynn_Collins, Samantha_Morton,...
Name: cast, dtype: object


In [132]:
for items in ['genres','cast','keywords']:
  check_len=list(movies.loc[movies[items].apply(len)==0].index)
  print(check_len)
  print(len(check_len))

[3976, 3997, 4073, 4110, 4123, 4299, 4320, 4391, 4406, 4419, 4464, 4510, 4556, 4568, 4572, 4575, 4577, 4587, 4617, 4628, 4639, 4663, 4680, 4687, 4720, 4722, 4807]
27
[2603, 3674, 3997, 4014, 4073, 4123, 4252, 4311, 4320, 4328, 4407, 4464, 4497, 4510, 4514, 4523, 4556, 4568, 4570, 4572, 4577, 4587, 4589, 4595, 4622, 4623, 4639, 4644, 4663, 4680, 4685, 4691, 4695, 4704, 4716, 4718, 4722, 4743, 4761, 4763, 4803]
41
[71, 83, 323, 381, 436, 605, 678, 857, 866, 881, 966, 967, 982, 1040, 1070, 1073, 1100, 1247, 1259, 1268, 1270, 1278, 1286, 1289, 1307, 1325, 1370, 1466, 1471, 1524, 1546, 1571, 1675, 1694, 1711, 1743, 1757, 1777, 1803, 1836, 1864, 1875, 1921, 1928, 1936, 1964, 1965, 1985, 1988, 2003, 2007, 2012, 2013, 2081, 2083, 2100, 2109, 2112, 2124, 2167, 2176, 2184, 2224, 2228, 2250, 2254, 2258, 2274, 2275, 2294, 2311, 2313, 2319, 2362, 2366, 2381, 2387, 2403, 2412, 2422, 2423, 2429, 2439, 2462, 2496, 2504, 2509, 2533, 2540, 2565, 2579, 2593, 2603, 2613, 2616, 2631, 2636, 2648, 2651, 2660

In [153]:
movies['empty_features']=(movies['genres'].apply(len)+movies['cast'].apply(len)+movies['keywords'].apply(len))
movies=movies[movies['empty_features']>0]
print(movies.shape)

(4791, 13)


In [134]:
print(movies['crew'][0])

[{'credit_id': '52fe48009251416c750aca23', 'department': 'Editing', 'gender': 0, 'id': 1721, 'job': 'Editor', 'name': 'Stephen E. Rivkin'}, {'credit_id': '539c47ecc3a36810e3001f87', 'department': 'Art', 'gender': 2, 'id': 496, 'job': 'Production Design', 'name': 'Rick Carter'}, {'credit_id': '54491c89c3a3680fb4001cf7', 'department': 'Sound', 'gender': 0, 'id': 900, 'job': 'Sound Designer', 'name': 'Christopher Boyes'}, {'credit_id': '54491cb70e0a267480001bd0', 'department': 'Sound', 'gender': 0, 'id': 900, 'job': 'Supervising Sound Editor', 'name': 'Christopher Boyes'}, {'credit_id': '539c4a4cc3a36810c9002101', 'department': 'Production', 'gender': 1, 'id': 1262, 'job': 'Casting', 'name': 'Mali Finn'}, {'credit_id': '5544ee3b925141499f0008fc', 'department': 'Sound', 'gender': 2, 'id': 1729, 'job': 'Original Music Composer', 'name': 'James Horner'}, {'credit_id': '52fe48009251416c750ac9c3', 'department': 'Directing', 'gender': 2, 'id': 2710, 'job': 'Director', 'name': 'James Cameron'}, 

In [158]:
director=movies['crew'].apply(
    lambda x:[i['name'].replace(' ','_')
    for i in x if i['job']=='Director']
    if isinstance(x,list) else [])
director=director.apply(lambda x:x[0] if len(x)>0 else "")


In [162]:
print(director[director.apply(lambda x:isinstance(x,str))].sum())
movies['director']=director.apply(lambda x:[x])


James_CameronGore_VerbinskiSam_MendesChristopher_NolanAndrew_StantonSam_RaimiByron_HowardJoss_WhedonDavid_YatesZack_SnyderBryan_SingerMarc_ForsterGore_VerbinskiGore_VerbinskiZack_SnyderAndrew_AdamsonJoss_WhedonRob_MarshallBarry_SonnenfeldPeter_JacksonMarc_WebbRidley_ScottPeter_JacksonChris_WeitzPeter_JacksonJames_CameronAnthony_RussoPeter_BergColin_TrevorrowSam_MendesSam_RaimiShane_BlackTim_BurtonBrett_RatnerDan_ScanlonMichael_BayMichael_BaySam_RaimiMarc_WebbJoseph_KosinskiJohn_LasseterMartin_CampbellLee_UnkrichMcGJames_WanMarc_ForsterBryan_SingerJ.J._AbramsBryan_SingerBaz_LuhrmannMike_NewellGuillermo_del_ToroMichael_BaySteven_SpielbergPeter_SohnBrenda_ChapmanJustin_LinAndrew_StantonBrett_RatnerRoland_EmmerichRobert_ZemeckisLilly_WachowskiDavid_YatesAndrew_AdamsonBryan_SingerChristopher_NolanPete_DocterConrad_VernonJon_FavreauMartin_ScorseseBarry_SonnenfeldRob_CohenDavid_AyerTom_ShadyacDoug_LimanKevin_ReynoldsStephen_SommersPete_DocterJon_FavreauJon_FavreauRupert_SandersRobert_Stromber

In [164]:
movies['overview']=movies['overview'].apply(lambda x:x.split())

AttributeError: 'list' object has no attribute 'split'

In [163]:
movies.head()

,genres,id,keywords,original_language,overview,release_date,revenue,title,vote_average,vote_count,cast,crew,empty_features,director
0,"[Action, Adventure, Fantasy, Science_Fiction]",19995,"[culture_clash, future, space_war, space_colon...",en,"[In, the, 22nd, century,, a, paraplegic, Marin...",2009-12-10,2787965087,Avatar,7.2,11800,"[Sam_Worthington, Zoe_Saldana, Sigourney_Weave...","[{'credit_id': '52fe48009251416c750aca23', 'de...",108,[James_Cameron]
1,"[Adventure, Fantasy, Action]",285,"[ocean, drug_abuse, exotic_island, east_india_...",en,"[Captain, Barbossa,, long, believed, to, be, d...",2007-05-19,961000000,Pirates of the Caribbean: At World's End,6.9,4500,"[Johnny_Depp, Orlando_Bloom, Keira_Knightley, ...","[{'credit_id': '52fe4232c3a36847f800b579', 'de...",53,[Gore_Verbinski]
2,"[Action, Adventure, Crime]",206647,"[spy, based_on_novel, secret_agent, sequel, mi...",en,"[A, cryptic, message, from, Bond’s, past, send...",2015-10-26,880674609,Spectre,6.3,4466,"[Daniel_Craig, Christoph_Waltz, Léa_Seydoux, R...","[{'credit_id': '54805967c3a36829b5002c41', 'de...",93,[Sam_Mendes]
3,"[Action, Crime, Drama, Thriller]",49026,"[dc_comics, crime_fighter, terrorist, secret_i...",en,"[Following, the, death, of, District, Attorney...",2012-07-16,1084939099,The Dark Knight Rises,7.6,9106,"[Christian_Bale, Michael_Caine, Gary_Oldman, A...","[{'credit_id': '52fe4781c3a36847f81398c3', 'de...",183,[Christopher_Nolan]
4,"[Action, Adventure, Science_Fiction]",49529,"[based_on_novel, mars, medallion, space_travel...",en,"[John, Carter, is, a, war-weary,, former, mili...",2012-03-07,284139100,John Carter,6.1,2124,"[Taylor_Kitsch, Lynn_Collins, Samantha_Morton,...","[{'credit_id': '52fe479ac3a36847f813eaa3', 'de...",46,[Andrew_Stanton]


In [165]:
len(movies['original_language'].unique())

37

In [166]:
from sklearn.preprocessing import LabelEncoder
enc=LabelEncoder()
og_lang=enc.fit_transform(movies['original_language'])
movies['og_lang']=og_lang

In [167]:
movies.head()

,genres,id,keywords,original_language,overview,release_date,revenue,title,vote_average,vote_count,cast,crew,empty_features,director,og_lang
0,"[Action, Adventure, Fantasy, Science_Fiction]",19995,"[culture_clash, future, space_war, space_colon...",en,"[In, the, 22nd, century,, a, paraplegic, Marin...",2009-12-10,2787965087,Avatar,7.2,11800,"[Sam_Worthington, Zoe_Saldana, Sigourney_Weave...","[{'credit_id': '52fe48009251416c750aca23', 'de...",108,[James_Cameron],7
1,"[Adventure, Fantasy, Action]",285,"[ocean, drug_abuse, exotic_island, east_india_...",en,"[Captain, Barbossa,, long, believed, to, be, d...",2007-05-19,961000000,Pirates of the Caribbean: At World's End,6.9,4500,"[Johnny_Depp, Orlando_Bloom, Keira_Knightley, ...","[{'credit_id': '52fe4232c3a36847f800b579', 'de...",53,[Gore_Verbinski],7
2,"[Action, Adventure, Crime]",206647,"[spy, based_on_novel, secret_agent, sequel, mi...",en,"[A, cryptic, message, from, Bond’s, past, send...",2015-10-26,880674609,Spectre,6.3,4466,"[Daniel_Craig, Christoph_Waltz, Léa_Seydoux, R...","[{'credit_id': '54805967c3a36829b5002c41', 'de...",93,[Sam_Mendes],7
3,"[Action, Crime, Drama, Thriller]",49026,"[dc_comics, crime_fighter, terrorist, secret_i...",en,"[Following, the, death, of, District, Attorney...",2012-07-16,1084939099,The Dark Knight Rises,7.6,9106,"[Christian_Bale, Michael_Caine, Gary_Oldman, A...","[{'credit_id': '52fe4781c3a36847f81398c3', 'de...",183,[Christopher_Nolan],7
4,"[Action, Adventure, Science_Fiction]",49529,"[based_on_novel, mars, medallion, space_travel...",en,"[John, Carter, is, a, war-weary,, former, mili...",2012-03-07,284139100,John Carter,6.1,2124,"[Taylor_Kitsch, Lynn_Collins, Samantha_Morton,...","[{'credit_id': '52fe479ac3a36847f813eaa3', 'de...",46,[Andrew_Stanton],7


In [174]:
movies['tag']=movies['genres']+movies['keywords']+movies['cast']+movies['director']+movies['overview']


In [188]:
title=movies['title'].apply(lambda x:x.split())
print(title.head())

0                                            [Avatar]
1    [Pirates, of, the, Caribbean:, At, World's, End]
2                                           [Spectre]
3                          [The, Dark, Knight, Rises]
4                                      [John, Carter]
Name: title, dtype: object


In [189]:
df=movies[['id','release_date','revenue','vote_average','vote_count','og_lang','tag']]

In [194]:
df.head()

,id,release_date,revenue,vote_average,vote_count,og_lang,tag,title
0,19995,2009-12-10,2787965087,7.2,11800,7,"[Action, Adventure, Fantasy, Science_Fiction, ...",[Avatar]
1,285,2007-05-19,961000000,6.9,4500,7,"[Adventure, Fantasy, Action, ocean, drug_abuse...","[Pirates, of, the, Caribbean:, At, World's, End]"
2,206647,2015-10-26,880674609,6.3,4466,7,"[Action, Adventure, Crime, spy, based_on_novel...",[Spectre]
3,49026,2012-07-16,1084939099,7.6,9106,7,"[Action, Crime, Drama, Thriller, dc_comics, cr...","[The, Dark, Knight, Rises]"
4,49529,2012-03-07,284139100,6.1,2124,7,"[Action, Adventure, Science_Fiction, based_on_...","[John, Carter]"


In [193]:
df['title']=title


/tmp/ipykernel_7227/2935662674.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['title']=title


In [195]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4791 entries, 0 to 4808
Data columns (total 8 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   id            4791 non-null   int64  
 1   release_date  4791 non-null   object 
 2   revenue       4791 non-null   int64  
 3   vote_average  4791 non-null   float64
 4   vote_count    4791 non-null   int64  
 5   og_lang       4791 non-null   int64  
 6   tag           4791 non-null   object 
 7   title         4791 non-null   object 
dtypes: float64(1), int64(4), object(3)
memory usage: 465.9+ KB
